# Assignment 4 — One- and Two-Sample Tests

**Course:** STA1

**Group members:** Bibek Paudel, Dora Buruncuk, Supendra Bogati

## Table of contents
1. [Part 1 — One-sample decisions](#part1)
2. [Part 2 — Independent coating processes](#part2)
3. [Part 3 — Paired building-energy measurements](#part3)
4. [Part 4 — Comparing two independent proportions](#part4)
5. [Part 5 — AI-use and conclusion](#part5)

In [1]:
import numpy as np
from scipy.stats import norm

<a id="part4"></a>
# Part 4 — Comparing two independent proportions

> Two independently assigned software configurations were evaluated using separate requests.
> Configuration A had 38 SLA failures among 300 requests; configuration B had 22 among 320. Define
> the risk difference as $p_A-p_B$.

## a)

> State the observational unit, binary outcome, populations, assumptions, comparison order, and
> two-sided hypotheses.

**Observational unit:** one request.

**Binary outcome:** whether a request is an SLA failure (yes or no).

**Populations:** all requests handled by configuration A and all requests handled by configuration
B under the evaluated conditions, with SLA-failure probabilities $p_A$ and $p_B$.

**Design:** two independent groups, because each request was evaluated with only one configuration.

**Assumptions:** requests are independent within and between groups, have the same failure
probability within each group, and represent the traffic each configuration would handle. Each
group needs enough expected failures and non-failures for the normal approximation; this is
checked in c).

**Comparison order:** the parameter is the risk difference $p_A - p_B$, fixed before the analysis.
A positive value means A fails more often. It is reported in percentage points.

**Hypotheses:** two-sided, at $\alpha = 0.05$:

$$H_0: p_A = p_B \qquad H_1: p_A \ne p_B$$

## b)

> Calculate both sample proportions, their difference in percentage points, and a 95% confidence
> interval using separate estimated variances.

$$\hat p_A = \frac{x_A}{n_A}, \qquad \hat p_B = \frac{x_B}{n_B}, \qquad
\text{SE} = \sqrt{\frac{\hat p_A(1-\hat p_A)}{n_A} + \frac{\hat p_B(1-\hat p_B)}{n_B}}$$

$$(\hat p_A - \hat p_B) \pm z_{0.975}\,\text{SE}$$

In [2]:
x_a, n_a = 38, 300   # configuration A: SLA failures, requests
x_b, n_b = 22, 320   # configuration B: SLA failures, requests

p_a = x_a / n_a
p_b = x_b / n_b
diff = p_a - p_b
se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)   # separate variances
z_crit = norm.ppf(0.975)
ci_lo, ci_hi = diff - z_crit * se, diff + z_crit * se

print(f"p_hat_A = {x_a}/{n_a} = {p_a:.5f} ({100 * p_a:.2f} %)")
print(f"p_hat_B = {x_b}/{n_b} = {p_b:.5f} ({100 * p_b:.2f} %)")
print(f"p_hat_A - p_hat_B = {diff:.5f} = {100 * diff:.2f} percentage points")
print(f"SE = {se:.5f}, z_0.975 = {z_crit:.4f}")
print(f"95% CI = [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"       = [{100 * ci_lo:.2f}, {100 * ci_hi:.2f}] percentage points")

p_hat_A = 38/300 = 0.12667 (12.67 %)
p_hat_B = 22/320 = 0.06875 (6.88 %)
p_hat_A - p_hat_B = 0.05792 = 5.79 percentage points
SE = 0.02385, z_0.975 = 1.9600
95% CI = [0.0112, 0.1047]
       = [1.12, 10.47] percentage points


**Result:** configuration A failed on 12.67 % of requests and B on 6.88 %. The difference is 5.79
percentage points, with a 95 % interval from 1.12 to 10.47 percentage points.

## c)

> Test $H_0:p_A=p_B$ with a two-proportion $z$ procedure. Report $z$, the two-sided $p$-value,
> decision, and contextual conclusion.

Under $H_0$ both configurations share one failure probability, so the test uses the pooled
proportion. The interval in b) does not assume $H_0$, so it uses separate variances.

$$\hat p = \frac{x_A + x_B}{n_A + n_B}, \qquad
\text{SE}_0 = \sqrt{\hat p(1-\hat p)\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}, \qquad
z = \frac{\hat p_A - \hat p_B}{\text{SE}_0}$$

In [3]:
alpha = 0.05

p_pool = (x_a + x_b) / (n_a + n_b)
se0 = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))   # pooled, because H0 is assumed
z_obs = diff / se0
p_two = 2 * norm.sf(abs(z_obs))

print(f"pooled proportion = {x_a + x_b}/{n_a + n_b} = {p_pool:.5f}")
print(f"expected failures under H0: A {n_a * p_pool:.1f}, B {n_b * p_pool:.1f}")
print(f"expected non-failures under H0: A {n_a * (1 - p_pool):.1f}, B {n_b * (1 - p_pool):.1f}")
print(f"SE0 = {se0:.5f}")
print(f"z = {diff:.5f} / {se0:.5f} = {z_obs:.3f}")
print(f"two-sided p = {p_two:.4f}")
print(f"decision at alpha = {alpha}:", "reject H0" if p_two < alpha else "fail to reject H0")

pooled proportion = 60/620 = 0.09677
expected failures under H0: A 29.0, B 31.0
expected non-failures under H0: A 271.0, B 289.0
SE0 = 0.02376
z = 0.05792 / 0.02376 = 2.438
two-sided p = 0.0148
decision at alpha = 0.05: reject H0


**Large-sample check:** the expected failures (29.0 and 31.0) and non-failures (271.0 and 289.0)
under $H_0$ are all well above 10, so the normal approximation is adequate.

**Decision:** $p = 0.0148 < 0.05$, so we reject $H_0$.

**Conclusion:** the data give evidence that the SLA-failure probabilities of the two configurations
differ. Configuration A is estimated to fail 5.79 percentage points more often than B, with a 95 %
interval from 1.12 to 10.47 percentage points.

## d)

> State what random sampling and random assignment would each support and why a significant
> difference alone is not enough for an unrestricted causal claim.

**Random sampling:** if the requests were randomly sampled from the target traffic, the result could
be generalised to that traffic.

**Random assignment:** if requests were randomly assigned to A or B, the configuration effect would
be separated from pre-existing differences between the two groups of requests, so the difference
could be attributed to the configuration.

**Why significance is not enough:** the small $p$-value only shows that a difference at least this
large would be unlikely if $p_A = p_B$ and the assumptions hold. It does not show how the requests
were selected or assigned. Without random assignment, other differences, such as request type or
time of day, could explain the gap. Without random sampling, the result covers only the requests
studied, not all traffic.